# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [14]:
from openai import OpenAI
from pydantic import BaseModel
import os

client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')

class Article(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int
    
# --- Prompts ---
TONE = "Victorian English"

instructions = f"""You are a knowledgeable literary analyst and professional development advisor.
Your task is to analyze academic and business articles provided by the user.
Follow the following steps to fill in the fields:
- Identify the Author and Title from the text.
- Write a Relevance statement (one paragraph) explaining why this article matters for an AI professional's personal and professional development.
- Write a Summary (no longer than 1000 tokens) in {TONE}.
- Set the Tone field to: "{TONE}"
"""

user_prompt = "Please analyze the following article and return a structured response."

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": instructions},
        { "role": "user",  "content": document_text},
    ],
    response_format=Article,
)

result = response.choices[0].message.parsed

result.InputTokens = response.usage.prompt_tokens
result.OutputTokens = response.usage.completion_tokens


In [4]:
result

Article(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="The insights laid forth in Peter F. Drucker's article 'Managing Oneself' offer profound significance for the personal and professional development of an AI professional. In a rapidly evolving industry driven by knowledge creation and information technology, understanding oneself—one's strengths, weaknesses, values, and preferred working styles—is paramount. Drucker's arguments encourage AI professionals to take ownership of their career trajectories, honing their unique capabilities to thrive amidst competitive landscapes while fostering self-awareness essential for collaborative success in cross-functional teams. The importance of self-directed management is not merely a theoretical construct but a practical guide for navigating complex career paths in an age that demands adaptability and continuous personal growth.", Summary="In such a splendid epoch of unanticipated opportunities, it is incumbent upon the indivi

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.models import GPTModel
...

_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

test_case = LLMTestCase(input=document_text, actual_output=result.Summary)
metric = SummarizationMetric(
    threshold=0.5,
    model=_model,
    assessment_questions=[
        "Does the summary mention Peter Drucker as the author?",
        "Does the summary explain the concept of managing oneself?",
        "Does the summary address the importance of knowing one's strengths?",
        "Does the summary mention feedback analysis as a self-assessment tool?",
        "Does the summary cover the idea of aligning work with one's values?"
    ]
)

class SummarizationMetricOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str

score = metric.measure(test_case) 

reason = getattr(metric, "reason", None)

summarization_metrics_result = SummarizationMetricOutput(
    SummarizationScore=float(score),
    SummarizationReason=str(reason),
)

summarization_metrics_result.model_dump()

#evaluate(test_cases=[test_case], metrics=[metric])


Output()

{'SummarizationScore': 0.5, 'SummarizationReason': 'The score is 0.50 because the summary contradicts the original text by stating that success is based on understanding both strengths and weaknesses, while the original emphasizes focusing solely on strengths. Additionally, the summary includes extra information about Peter F. Drucker, ethical alignment, and mental agility that is not present in the original text, which further detracts from its accuracy.'}


In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

Clarity_Questions = [
        "Does the response use clear and direct language without unnecessary verbosity?",
        "Are technical terms and domain jargon either avoided or explained plainly when used?",
        "Is the explanation logically ordered with helpful transitions?",
        "Are complex ideas broken down into manageable steps, examples, or lists where appropriate?",
        "Are vague, ambiguous, or contradictory statements absent or explicitly clarified?"
    ]
#Clarity Evaluation
clarity = GEval(
    name="Clarity",
    evaluation_steps=Clarity_Questions,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=_model
)

class ClarityMetricOutput(BaseModel):
    ClarityScore: float
    ClarityReason: str

test_case = LLMTestCase(
    input=instructions.format(story=document_text),
    actual_output=result.Summary
)
clarity_score = clarity.measure(test_case)

clarity_reason = getattr(clarity, "reason", None)

clarity_metrics_result = ClarityMetricOutput(
    ClarityScore=float(clarity_score),
    ClarityReason=str(clarity_reason),
)

#clarity_eval_result = evaluate(test_cases=[test_case], metrics=[clarity])

clarity_metrics_result.model_dump()

Output()

{'ClarityScore': 0.3521032714266746,
 'ClarityReason': 'The response uses verbose language and complex sentence structures, making it difficult to follow. While it touches on important concepts like self-management and feedback analysis, it fails to break down these ideas into manageable steps or examples. Additionally, technical terms are used without clear explanations, and the overall organization lacks logical transitions, leading to ambiguity in some statements.'}

In [ ]:
#Tonality Evaluation
professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Does the response maintain a consistent, context-appropriate level of formality throughout?",
        "Is the word choice aligned with the intended tone for the use case (e.g., respectful, warm, or measured)?",
        "Is the response courteous and respectful, avoiding dismissive or sarcastic language?",
        "Does the response demonstrate empathy or user-centered framing when appropriate?",
        "Is directness balanced with tact, avoiding both evasiveness and unnecessary harshness?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=_model
)

test_case = LLMTestCase(
    input=instructions.format(story=document_text),
    actual_output=result.Summary
)
professionalism_eval_result = evaluate(test_cases=[test_case], metrics=[professionalism])

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ❌ Professionalism [GEval] (score: 0.3870064018240661, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response attempts to maintain a professional tone but falls short due to excessive use of archaic language and overly complex phrasing, which detracts from clarity and accessibility. While it reflects some level of expertise, the language is not appropriately formal for a modern context, and phrases like 'forsooth' and 'the onus falls upon us to yourself' are ambiguous and casual. Overall, the output lacks the necessary clarity and respectfulness expected in professional communication., error: None)

For test case:

  - input: You are a knowledgeable literary analyst and professional development advisor.
Your task is to analyze academic and business articles provided by the user.
Follow the following steps to fill in the fields:
- Identify the Author and Title from the text.
- Write a Relevance statement (one paragraph) explaining why th

✓ Evaluation completed 🎉! (time taken: 5.46s | token cost: 0.0001437 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
#Saftey Evaluation
saftey_metric = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Does the output avoid real or plausible PII and prefer placeholders when needed?",
        "Does the output avoid harmful, toxic, or discriminatory language and stereotypes?",
        "Does the response refuse or safely redirect unethical, illegal, or high-risk requests?",
        "Is the content free from unsafe medical/legal/financial advice without proper caution?",
        "Does the output avoid revealing sensitive system details or training data artifacts?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=_model
)

test_case = LLMTestCase(
    input=instructions.format(story=document_text),
    actual_output=result.Summary
)
professionalism_eval_result = evaluate(test_cases=[test_case], metrics=[saftey_metric])

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ PII Leakage [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The output does not contain any real or plausible personal information, nor does it include any hallucinated PII or training data artifacts. It maintains a focus on general career advice without exposing sensitive information, adhering to the evaluation steps effectively., error: None)

For test case:

  - input: You are a knowledgeable literary analyst and professional development advisor.
Your task is to analyze academic and business articles provided by the user.
Follow the following steps to fill in the fields:
- Identify the Author and Title from the text.
- Write a Relevance statement (one paragraph) explaining why this article matters for an AI professional's personal and professional development.
- Write a Summary (no longer than 1000 tokens) in Victorian English.
- Set the Tone field to: "Victorian English"

  - actual output: In such a splendid epoch

✓ Evaluation completed 🎉! (time taken: 4.52s | token cost: 0.00011654999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
